# YOLO Training - Resume from Epoch 56
This notebook resumes your YOLO training from epoch 56/300 using FREE Google Colab GPU.

**Current Status:**
- Checkpoint: Epoch 56/300 (18.7% complete)
- Remaining: 244 epochs
- Estimated Time: 8-10 hours on T4 GPU
- Cost: $0 (FREE!)

## Step 1: Check GPU Availability

In [ ]:
import torch
print(f"🎮 GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"🎮 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  GPU not available. Go to Runtime → Change runtime type → Select T4 GPU")

## Step 2: Install Dependencies

In [ ]:
!pip install -q ultralytics
print("✅ Dependencies installed")

## Step 3: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted")

## Step 4: Set Up Dataset
This will extract your training and validation data from the ZIP files you uploaded.

In [ ]:
import os
import zipfile

# Create dataset directory
os.makedirs('/content/yolo_dataset', exist_ok=True)
os.chdir('/content/yolo_dataset')

# Extract training data
print("📦 Extracting training images...")
with zipfile.ZipFile('/content/drive/MyDrive/yolo-training/train_images.zip', 'r') as zip_ref:
    zip_ref.extractall('train/images')

print("📦 Extracting training labels...")
with zipfile.ZipFile('/content/drive/MyDrive/yolo-training/train_labels.zip', 'r') as zip_ref:
    zip_ref.extractall('train/labels')

# Extract validation data
print("📦 Extracting validation images...")
with zipfile.ZipFile('/content/drive/MyDrive/yolo-training/val_images.zip', 'r') as zip_ref:
    zip_ref.extractall('val/images')

print("📦 Extracting validation labels...")
with zipfile.ZipFile('/content/drive/MyDrive/yolo-training/val_labels.zip', 'r') as zip_ref:
    zip_ref.extractall('val/labels')

# Verify dataset
train_images = len(os.listdir('train/images'))
val_images = len(os.listdir('val/images'))
print(f"\n✅ Dataset ready:")
print(f"   Train images: {train_images}")
print(f"   Val images: {val_images}")

## Step 5: Create data.yaml Configuration

In [ ]:
# Create data.yaml
data_yaml = """path: /content/yolo_dataset
train: train/images
val: val/images

names:
  0: general_damage
  1: cracks
  2: mold
  3: water_damage
  4: structural_damage
  5: electrical_issues
  6: plumbing_issues
  7: roofing_damage
  8: window_damage
  9: door_damage
  10: floor_damage
  11: wall_damage
  12: ceiling_damage
  13: hvac_issues
  14: insulation_issues
"""

with open('data.yaml', 'w') as f:
    f.write(data_yaml)

print("✅ data.yaml created")

## Step 6: Load Checkpoint and Resume Training
This will resume from epoch 56 and train to epoch 300.

In [ ]:
from ultralytics import YOLO
import torch
import shutil

# Copy checkpoint from Google Drive
checkpoint_path = '/content/drive/MyDrive/yolo-training/last.pt'
print(f"📥 Loading checkpoint: {checkpoint_path}")

# FIX FOR PyTorch 2.6 weights_only security change
print("🔧 Cleaning checkpoint (fixing PyTorch 2.6 compatibility)...")

# Add safe globals for Ultralytics classes
torch.serialization.add_safe_globals([
    'ultralytics.nn.tasks.DetectionModel',
    'ultralytics.nn.modules',
    'collections.OrderedDict'
])

# Load checkpoint with weights_only=False (safe since we trust our own checkpoint)
checkpoint = torch.load(checkpoint_path, map_location='cuda:0', weights_only=False)

# Remove GradScaler state if present (causes error when resuming on GPU)
if 'scaler' in checkpoint:
    del checkpoint['scaler']
    print("   ✓ Removed GradScaler state")

# Force AMP to False in checkpoint args to prevent conflicts
if 'args' in checkpoint and isinstance(checkpoint['args'], dict):
    checkpoint['args']['amp'] = False
    print("   ✓ Set checkpoint amp=False")

# Save cleaned checkpoint
cleaned_checkpoint_path = '/content/last_cleaned.pt'
torch.save(checkpoint, cleaned_checkpoint_path)
print(f"✅ Cleaned checkpoint saved to: {cleaned_checkpoint_path}")

# Load model from cleaned checkpoint
model = YOLO(cleaned_checkpoint_path)

print("✅ Checkpoint loaded successfully!")
print("🚀 Starting training from epoch 57...")
print("⏱️  Estimated time: 8-10 hours")
print("💾 Progress will be saved to Google Drive every 10 epochs")

In [ ]:
# Resume training with FIXED parameters
results = model.train(
    data='data.yaml',
    epochs=300,
    imgsz=640,
    batch=16,          # Optimal for T4 GPU
    device=0,          # Use GPU
    patience=50,
    save=True,
    save_period=10,    # Save checkpoint every 10 epochs
    cache=True,        # Cache images for faster training
    amp=False,         # ⚠️ CRITICAL: Set to False to avoid GradScaler error
    project='/content/drive/MyDrive/yolo-training/runs',
    name='colab_gpu_run',
    exist_ok=True,
    resume=False,      # Set to False since we're loading manually
    plots=True,        # Generate training plots
    val=True,          # Run validation
    
    # Hyperparameters (from your original config)
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0,
    copy_paste=0.0,
)

## Step 7: Copy Final Model to Google Drive

In [ ]:
import shutil

# Copy best model
best_model_src = '/content/drive/MyDrive/yolo-training/runs/colab_gpu_run/weights/best.pt'
best_model_dst = '/content/drive/MyDrive/yolo-training/best_model_final.pt'

shutil.copy(best_model_src, best_model_dst)

print("✅ Training Complete!")
print(f"📊 Final model saved to: {best_model_dst}")
print(f"📈 Training results: /content/drive/MyDrive/yolo-training/runs/colab_gpu_run/")

## Step 8: Display Results

In [ ]:
from IPython.display import Image, display

# Display training results
results_dir = '/content/drive/MyDrive/yolo-training/runs/colab_gpu_run'

print("📊 Training Results:")
print("="*50)

# Display confusion matrix
print("\n📈 Confusion Matrix:")
display(Image(filename=f'{results_dir}/confusion_matrix.png'))

# Display results
print("\n📈 Training Curves:")
display(Image(filename=f'{results_dir}/results.png'))

# Display predictions
print("\n🖼️  Sample Predictions:")
display(Image(filename=f'{results_dir}/val_batch0_pred.jpg'))

## Step 9: Test the Model

In [ ]:
# Load best model and run validation
best_model = YOLO('/content/drive/MyDrive/yolo-training/best_model_final.pt')

# Validate on test set
metrics = best_model.val()

print("\n🎯 Final Model Performance:")
print("="*50)
print(f"mAP@50: {metrics.box.map50:.3f}")
print(f"mAP@50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")
print("="*50)

if metrics.box.map50 > 0.70:
    print("✅ Model meets target performance (mAP@50 > 70%)")
else:
    print(f"⚠️  Model below target. Current: {metrics.box.map50:.1%}, Target: 70%")

## 🎉 Training Complete!

Your model is now trained and saved to Google Drive:
- **Best Model**: `/content/drive/MyDrive/yolo-training/best_model_final.pt`
- **Results**: `/content/drive/MyDrive/yolo-training/runs/colab_gpu_run/`

### Next Steps:
1. Download `best_model_final.pt` from Google Drive
2. Test it locally with your validation images
3. Deploy to production (Supabase storage)

### Model Info:
- **Training**: Epochs 1-300 (resumed from 56)
- **GPU**: Tesla T4
- **Time**: ~8-10 hours
- **Cost**: $0 (FREE!)